# Experiment 04 — Deep Architecture Comparison

**Extension of `03_feature_ablation.ipynb`** — adds 6 new architectures under the same E0–E5 experimental design.

| Experiment | Features | Lookback |
|------------|----------|----------|
| E0 | Load only | 24 h |
| E1 | Load + Weather | 24 h |
| E2 | Load + Temporal | 24 h |
| E3 | Load + Lag-24 + Lag-168 | 24 h |
| E4 | Load + Weather + Temporal + Lags | 24 h |
| E5 | Load + Weather + Temporal + Lags | 168 h |

**New architectures:** Transformer, TFT, TCN, NBEATS-MV, Informer, PatchTST

**Fair comparison:**
- All models receive identical feature representations per experiment.
- Model selection uses **validation MAPE only** — test set is held-out.
- All predictions saved in long-format CSV for post-hoc analysis.
- Resume/cache mode: set `REUSE_EXISTING_RESULTS=True` to skip already-trained models.

**Key design decisions documented in `src/models/tft.py`, `src/models/nbeats.py`.**


In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────
from pathlib import Path
import subprocess, sys, os

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    # Try local workspace
    local_root = Path(".").resolve()
    # Walk up to find the src/ directory
    for p in [local_root, local_root.parent, local_root.parent.parent]:
        if (p / "src").exists() and (p / "configs").exists():
            PROJECT_ROOT = p
            break
    else:
        # Kaggle: clone from GitHub
        PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")
        subprocess.run(
            ["git", "clone",
             "https://github.com/AlvinHarist/stlf-entso-2026.git",
             str(PROJECT_ROOT)],
            check=True,
        )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
# ── Cell 2: Imports & Configuration ──────────────────────────────────
import time
import json
import yaml
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

import tensorflow as tf
print(f"TensorFlow version : {tf.__version__}")
print(f"GPUs Available     : {len(tf.config.list_physical_devices('GPU'))}")

from sklearn.linear_model import LinearRegression

from src.utils.seed import set_seed
from src.data.load_data import load_dataset
from src.data.preprocessing import (
    chronological_split,
    fit_preprocessor,
    transform_data,
    inverse_y,
    create_temporal_features,
    create_lag_features,
    TEMPORAL_FEATURE_COLS,
    LAG_FEATURE_COLS_DEFAULT,
)
from src.data.windowing import create_train_windows, create_evaluation_windows
from src.models.lstm import build_lstm
from src.models.bilstm import build_bilstm
from src.models.transformer import build_transformer
from src.models.tft import build_tft, TFT_KNOWN_FUTURE_COLS
from src.models.tcn import build_tcn
from src.models.nbeats import build_nbeats
from src.models.informer import build_informer
from src.models.patchtst import build_patchtst
from src.training.trainer import train_model
from src.evaluation.point_metrics import (
    compute_all_metrics,
    compute_naive_baselines,
    mae, rmse, mape, smape,
)

CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]                      # 42
TARGET_COL = config["data"]["target_col"]
WEATHER    = config["data"]["weather_features"]
UNITS      = config["model"]["units"]            # 64
DROPOUT    = config["model"]["dropout"]          # 0.0 → we use 0.1 for new models
LR         = config["model"]["learning_rate"]   # 0.001
EPOCHS     = config["training"]["epochs"]        # 200
BATCH_SIZE = config["training"]["batch_size"]   # 32
PATIENCE   = config["training"]["patience"]     # 15
HORIZON    = 24

# ── Pipeline control ─────────────────────────────────────────────────
RUN_EXPERIMENTS      = True
REUSE_EXISTING_RESULTS = True  # Set False to force retrain all

# ── Hyperparameters for new architectures ────────────────────────────
NEW_MODEL_HPARAMS = {
    "Transformer": dict(d_model=64, n_heads=4, n_layers=2, dff=128, dropout=0.1, learning_rate=LR),
    "TFT":         dict(hidden_dim=64, n_heads=4, n_lstm_layers=2, dropout=0.1, learning_rate=LR),
    "TCN":         dict(filters=64, kernel_size=3, n_blocks=4, dropout=0.1, learning_rate=LR),
    "NBEATS-MV":   dict(n_stacks=2, n_blocks_per_stack=3, hidden_units=256, layers_per_block=4, learning_rate=LR),
    "Informer":    dict(d_model=64, n_heads=4, n_layers=2, dff=128, dropout=0.1, use_distilling=True, learning_rate=LR),
    "PatchTST":    dict(patch_len=None, stride=None, d_model=64, n_heads=4, n_layers=2, dff=128, dropout=0.1, learning_rate=LR),
}

# ── Results directories ───────────────────────────────────────────────
RESULTS_DIR  = PROJECT_ROOT / "results" / "deep_architectures"
PRED_DIR     = RESULTS_DIR / "predictions"
METRICS_DIR  = RESULTS_DIR / "metrics"

for d in [PRED_DIR / "validation", PRED_DIR / "test", METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dataset    : {TARGET_COL}")
print(f"Results    : {RESULTS_DIR}")
print(f"Seed       : {SEED}")
print(f"Horizon    : {HORIZON} h")
print(f"Resume mode: REUSE_EXISTING_RESULTS={REUSE_EXISTING_RESULTS}")

In [ ]:
# ── Cell 3: Data Loading & Feature Engineering ────────────────────────
set_seed(SEED)

# Locate data file
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    for fallback in [
        PROJECT_ROOT / config["data"]["path"],
        PROJECT_ROOT / "df_combined_AT.csv",
        PROJECT_ROOT / "df_combined_clean_AT.csv",
    ]:
        if fallback.exists():
            DATA_PATH = fallback
            break
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

print(f"Loading data from: {DATA_PATH}")
df_raw = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
print(f"Raw dataset: {df_raw.shape}  {df_raw.index.min()} --> {df_raw.index.max()}")

# Feature engineering
df_feat = create_temporal_features(df_raw)
df_feat = create_lag_features(df_feat, TARGET_COL, lags=[24, 168])

print(f"Feature-engineered dataset: {df_feat.shape}")
print(f"Columns: {list(df_feat.columns)}")

# Sanity check
sample_t = 200
assert abs(df_feat["lag_168"].iloc[sample_t] - df_feat[TARGET_COL].iloc[sample_t - 168]) < 1e-6, \
    "Lag feature sanity check FAILED!"
print("Lag sanity check: PASSED")

In [ ]:
# ── Cell 4: Chronological Split ────────────────────────────────────────
train_df, val_df, test_df = chronological_split(
    df_feat,
    train_ratio=config["split"]["train_ratio"],
    val_ratio=config["split"]["val_ratio"],
)

full_series_mw = df_feat[TARGET_COL].values
val_start_idx  = len(train_df)
test_start_idx = len(train_df) + len(val_df)

# Timestamps for prediction CSVs
df_timestamps = df_feat.index  # DatetimeIndex

# Leakage assertions
assert train_df.index.max() < val_df.index.min(), "Train/val boundary violation!"
assert val_df.index.max() < test_df.index.min(), "Val/test boundary violation!"
print("Split boundary assertions: PASSED")
print(f"Train: {len(train_df)} rows | Val: {len(val_df)} rows | Test: {len(test_df)} rows")

In [ ]:
# ── Cell 5: Helper Functions ───────────────────────────────────────────

# ── 5A: Prediction saving in long-format ──────────────────────────────
def get_window_timestamps(start_idx: int, n_windows: int, lookback: int, horizon: int):
    """Return forecast_origin and target_timestamps for each window."""
    origins = []
    targets = []
    for w in range(n_windows):
        # The forecast origin is the last input time step
        origin_row = start_idx + w + lookback - 1
        if origin_row < len(df_timestamps):
            origin_ts = df_timestamps[origin_row]
        else:
            origin_ts = pd.NaT
        origins.append(origin_ts)
        window_targets = []
        for h in range(horizon):
            target_row = start_idx + w + lookback + h
            if target_row < len(df_timestamps):
                window_targets.append(df_timestamps[target_row])
            else:
                window_targets.append(pd.NaT)
        targets.append(window_targets)
    return origins, targets


def save_predictions_long(
    pred_mw: np.ndarray,
    actual_mw: np.ndarray,
    exp_name: str,
    model_name: str,
    split: str,
    lookback: int,
    split_start_idx: int,
    overwrite: bool = True,
) -> Path:
    """
    Save predictions in long format:
    forecast_origin, target_timestamp, horizon, true, pred, window_mape

    Performs 7 consistency checks before saving.

    Returns the path to the saved CSV.
    """
    n_windows, h_size = pred_mw.shape
    assert h_size == HORIZON, f"Expected horizon={HORIZON}, got {h_size}"

    out_dir = PRED_DIR / split / exp_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{model_name}_lb{lookback}.csv"

    if out_path.exists() and not overwrite:
        print(f"  [SKIP] {out_path.name} already exists (overwrite=False)")
        return out_path

    # ── Consistency checks ─────────────────────────────────────────
    # Check 1: n_windows
    assert n_windows > 0, "n_windows must be > 0"

    # Check 2: horizon = 24
    assert pred_mw.shape[1] == HORIZON, f"Prediction shape[1]={pred_mw.shape[1]} != HORIZON={HORIZON}"

    # Check 3: no NaN
    assert not np.any(np.isnan(pred_mw)), "NaN detected in predictions!"
    assert not np.any(np.isnan(actual_mw)), "NaN detected in actuals!"

    # Check 4: predictions are in reasonable MW scale (not scaled 0-1)
    if pred_mw.max() <= 1.5 and pred_mw.min() >= -0.5:
        import warnings
        warnings.warn("Predictions appear to be in scaled space (max<=1.5). "
                      "Did you forget inverse_y()?", UserWarning)

    # Compute per-window MAPE
    window_mapes = np.array([
        mape(actual_mw[i], pred_mw[i]) for i in range(n_windows)
    ])

    # Check 5: window_mape consistency with overall MAPE
    overall_mape_flat = mape(actual_mw.ravel(), pred_mw.ravel())
    mean_window_mape  = float(np.nanmean(window_mapes))
    if abs(overall_mape_flat - mean_window_mape) > 5.0:  # allow 5% difference
        print(f"  [WARNING] overall MAPE={overall_mape_flat:.3f}% vs "
              f"mean window MAPE={mean_window_mape:.3f}%  (diff >{5}%)")

    # Build long-format rows
    origins, target_tss = get_window_timestamps(
        split_start_idx, n_windows, lookback, HORIZON
    )

    rows = []
    for w in range(n_windows):
        w_mape = window_mapes[w]
        for h in range(HORIZON):
            rows.append({
                "forecast_origin":   origins[w],
                "target_timestamp":  target_tss[w][h],
                "horizon":           h + 1,
                "true":              float(actual_mw[w, h]),
                "pred":              float(pred_mw[w, h]),
                "window_mape":       float(w_mape),
            })

    df_long = pd.DataFrame(rows)

    # Check 6: correct total rows
    expected_rows = n_windows * HORIZON
    assert len(df_long) == expected_rows, \
        f"Expected {expected_rows} rows, got {len(df_long)}"

    # Check 7: timestamps are not all NaT
    n_valid_ts = df_long["forecast_origin"].notna().sum()
    if n_valid_ts == 0:
        print("  [WARNING] All forecast_origin timestamps are NaT — check split index.")

    df_long.to_csv(out_path, index=False)
    print(f"  [SAVED] {out_path}  ({n_windows} windows × {HORIZON} steps = {len(df_long)} rows)")
    return out_path


def load_predictions(experiment: str, model: str, split: str, lookback: int | None = None) -> pd.DataFrame:
    """
    Load previously saved predictions from CSV WITHOUT retraining.

    Parameters
    ----------
    experiment : str  e.g. 'E5_full_168'
    model      : str  e.g. 'Transformer'
    split      : str  'validation' or 'test'
    lookback   : int or None  if None, auto-detects from filename

    Returns
    -------
    pd.DataFrame with columns:
        forecast_origin, target_timestamp, horizon, true, pred, window_mape
    """
    search_dir = PRED_DIR / split / experiment
    if lookback is not None:
        path = search_dir / f"{model}_lb{lookback}.csv"
    else:
        matches = list(search_dir.glob(f"{model}_lb*.csv"))
        if len(matches) == 0:
            raise FileNotFoundError(
                f"No prediction file found for {experiment}/{model} in {split}"
            )
        path = matches[0]  # take first match
    df = pd.read_csv(path, parse_dates=["forecast_origin", "target_timestamp"])
    return df


def prediction_file_exists(exp_name: str, model_name: str,
                            split: str, lookback: int) -> bool:
    path = PRED_DIR / split / exp_name / f"{model_name}_lb{lookback}.csv"
    return path.exists()


# ── 5B: Metric record saving ────────────────────────────────────────
METRIC_RECORDS = []  # appended after each experiment run

def append_metric_record(exp_name, model_name, lookback, split, metrics_dict,
                          best_epoch="N/A", train_time_s=None, n_params=None):
    rec = {
        "experiment":   exp_name,
        "model":        model_name,
        "lookback":     lookback,
        "split":        split,
        "best_epoch":   best_epoch,
        "train_time_s": train_time_s,
        "n_params":     n_params,
    }
    rec.update(metrics_dict)
    METRIC_RECORDS.append(rec)


# ── 5C: Model builder dispatch ───────────────────────────────────────
def build_model(model_name: str, lookback: int, n_features: int,
                horizon: int, feature_cols: list) -> object:
    """
    Build a compiled model for the given model_name.
    Returns a Keras model or sklearn estimator.
    """
    if model_name == "LinearRegression":
        return LinearRegression()

    elif model_name == "LSTM":
        return build_lstm(lookback, n_features, horizon, UNITS, DROPOUT, LR)

    elif model_name == "BiLSTM":
        return build_bilstm(lookback, n_features, horizon, UNITS, DROPOUT, LR)

    elif model_name == "Transformer":
        p = NEW_MODEL_HPARAMS["Transformer"]
        return build_transformer(lookback, n_features, horizon, **p)

    elif model_name == "TFT":
        p = {k: v for k, v in NEW_MODEL_HPARAMS["TFT"].items()}
        return build_tft(lookback, n_features, horizon,
                         feature_cols=feature_cols, **p)

    elif model_name == "TCN":
        p = NEW_MODEL_HPARAMS["TCN"]
        return build_tcn(lookback, n_features, horizon, **p)

    elif model_name == "NBEATS-MV":
        p = NEW_MODEL_HPARAMS["NBEATS-MV"]
        return build_nbeats(lookback, n_features, horizon, **p)

    elif model_name == "Informer":
        p = NEW_MODEL_HPARAMS["Informer"]
        return build_informer(lookback, n_features, horizon, **p)

    elif model_name == "PatchTST":
        p = NEW_MODEL_HPARAMS["PatchTST"]
        return build_patchtst(lookback, n_features, horizon, **p)

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


# ── 5D: Core run_experiment function ─────────────────────────────────
def run_experiment(
    exp_name: str,
    feature_cols: list,
    lookback: int,
    model_name: str,
    evaluate_on_test: bool = False,
):
    """
    Run one (experiment × model) combination.

    Returns a result dict with val metrics and (optionally) test metrics.
    Saves long-format prediction CSVs automatically.

    If REUSE_EXISTING_RESULTS=True and prediction CSV already exists,
    loads from disk without retraining.
    """
    set_seed(SEED)

    # ── Check for cached results ──────────────────────────────────────
    val_exists  = prediction_file_exists(exp_name, model_name, "validation", lookback)
    test_exists = prediction_file_exists(exp_name, model_name, "test", lookback)

    if REUSE_EXISTING_RESULTS and val_exists:
        print(f"  [CACHE] {exp_name}/{model_name} val predictions already exist — loading.")
        df_val_pred = load_predictions(exp_name, model_name, "validation", lookback)
        pred_va_mw  = df_val_pred["pred"].values.reshape(-1, HORIZON)
        actual_va_mw = df_val_pred["true"].values.reshape(-1, HORIZON)
        val_mets = compute_all_metrics(actual_va_mw, pred_va_mw)

        result = {
            "exp": exp_name, "model": model_name,
            "lookback": lookback, "best_epoch": "cached",
            "val_metrics": val_mets,
            "pred_va_mw": pred_va_mw,
            "actual_va_mw": actual_va_mw,
        }

        if evaluate_on_test and test_exists:
            df_test_pred = load_predictions(exp_name, model_name, "test", lookback)
            pred_te_mw   = df_test_pred["pred"].values.reshape(-1, HORIZON)
            actual_te_mw = df_test_pred["true"].values.reshape(-1, HORIZON)
            test_mets = compute_all_metrics(actual_te_mw, pred_te_mw)
            result["test_metrics"] = test_mets
            result["pred_te_mw"]   = pred_te_mw
            result["actual_te_mw"] = actual_te_mw
            print(f"  [CACHE] {exp_name}/{model_name} test predictions loaded.")

        print(f"  [CACHE] VAL  MAE={val_mets['MAE']:.2f}  MAPE={val_mets['MAPE']:.2f}%")
        return result

    # ── Preprocessing (train-only fit) ────────────────────────────────
    use_yj = any(c in feature_cols for c in config["preprocessing"]["skewed_cols"])
    skewed = config["preprocessing"]["skewed_cols"] if use_yj else None

    preprocessor = fit_preprocessor(
        train_df, TARGET_COL, feature_cols,
        skewed_cols=skewed, use_yeojohnson=use_yj,
    )

    X_tr, y_tr = transform_data(train_df, preprocessor)
    X_va, y_va = transform_data(val_df,   preprocessor)
    X_te, y_te = transform_data(test_df,  preprocessor)

    # ── Windowing ─────────────────────────────────────────────────────
    Xw_tr, yw_tr = create_train_windows(X_tr, y_tr, lookback, HORIZON)
    Xw_va, yw_va = create_evaluation_windows(X_va, y_va, X_tr, y_tr, lookback, HORIZON)
    Xw_te, yw_te = create_evaluation_windows(X_te, y_te, X_va, y_va, lookback, HORIZON)

    print(f"  [{exp_name}/{model_name}] Windows  tr={Xw_tr.shape}  va={Xw_va.shape}  te={Xw_te.shape}")

    n_features = Xw_tr.shape[2]

    # ── Build & Train ──────────────────────────────────────────────────
    best_epoch  = "N/A"
    n_params    = None
    train_time  = None

    t0 = time.time()

    model = build_model(model_name, lookback, n_features, HORIZON, feature_cols)

    if model_name == "LinearRegression":
        model.fit(Xw_tr.reshape(Xw_tr.shape[0], -1), yw_tr)
        pred_va_scaled = model.predict(Xw_va.reshape(Xw_va.shape[0], -1))
        pred_te_scaled = model.predict(Xw_te.reshape(Xw_te.shape[0], -1))
        n_params = model.coef_.size + model.intercept_.size
    else:
        n_params = model.count_params()
        tr_res = train_model(
            model, Xw_tr, yw_tr, Xw_va, yw_va,
            EPOCHS, BATCH_SIZE, PATIENCE, verbose=0,
        )
        best_epoch = tr_res["best_epoch"]

        t_infer = time.time()
        pred_va_scaled = model.predict(Xw_va, verbose=0)
        pred_te_scaled = model.predict(Xw_te, verbose=0)
        infer_time_ms  = (time.time() - t_infer) * 1000 / len(Xw_va)

    train_time = time.time() - t0

    # ── Inverse Transform ──────────────────────────────────────────────
    pred_va_mw   = inverse_y(pred_va_scaled, preprocessor)
    actual_va_mw = inverse_y(yw_va,          preprocessor)
    pred_te_mw   = inverse_y(pred_te_scaled, preprocessor)
    actual_te_mw = inverse_y(yw_te,          preprocessor)

    # ── Metrics ────────────────────────────────────────────────────────
    val_mets  = compute_all_metrics(actual_va_mw, pred_va_mw)
    test_mets = compute_all_metrics(actual_te_mw, pred_te_mw)

    # ── Save Validation Predictions (long format) ─────────────────────
    save_predictions_long(
        pred_va_mw, actual_va_mw,
        exp_name, model_name, "validation", lookback,
        split_start_idx=val_start_idx,
    )

    # ── Save Test Predictions (always, for post-selection analysis) ───
    save_predictions_long(
        pred_te_mw, actual_te_mw,
        exp_name, model_name, "test", lookback,
        split_start_idx=test_start_idx,
    )

    # ── Append metric records ──────────────────────────────────────────
    append_metric_record(exp_name, model_name, lookback, "validation",
                         val_mets, best_epoch, train_time, n_params)
    append_metric_record(exp_name, model_name, lookback, "test",
                         test_mets, best_epoch, train_time, n_params)

    print(f"  [{exp_name}/{model_name}] VAL   MAE={val_mets['MAE']:.2f}  MAPE={val_mets['MAPE']:.2f}%")
    print(f"  [{exp_name}/{model_name}] TEST  MAE={test_mets['MAE']:.2f}  MAPE={test_mets['MAPE']:.2f}%")
    print(f"  [{exp_name}/{model_name}] n_params={n_params}  train_time={train_time:.1f}s")

    result = {
        "exp": exp_name, "model": model_name,
        "feature_cols": feature_cols, "lookback": lookback,
        "best_epoch": best_epoch, "n_params": n_params,
        "train_time_s": train_time,
        "val_metrics":  val_mets,
        "test_metrics": test_mets,
        "pred_va_mw":   pred_va_mw,
        "actual_va_mw": actual_va_mw,
        "pred_te_mw":   pred_te_mw,
        "actual_te_mw": actual_te_mw,
    }
    return result


# ── 5E: Naive baselines ────────────────────────────────────────────────
def run_naive_baselines(split="val"):
    """Compute naive baselines (Persistence, Daily, Weekly)."""
    if split == "val":
        start_idx = val_start_idx
        n_windows = len(val_df) - HORIZON + 1
    else:
        start_idx = test_start_idx
        n_windows = len(test_df) - HORIZON + 1

    y_actual = np.array([
        full_series_mw[start_idx + w : start_idx + w + HORIZON]
        for w in range(n_windows)
    ])
    return compute_naive_baselines(y_actual, full_series_mw, start_idx, HORIZON)


print("Helper functions defined.")

In [ ]:
# ── Cell 6: Experiment Configurations ─────────────────────────────────
EXPERIMENTS = [
    {
        "name":         "E0_load_only",
        "feature_cols": [TARGET_COL],
        "lookback":     24,
        "description":  "Load only (univariate baseline)",
    },
    {
        "name":         "E1_weather",
        "feature_cols": [TARGET_COL] + WEATHER,
        "lookback":     24,
        "description":  "Load + weather variables",
    },
    {
        "name":         "E2_temporal",
        "feature_cols": [TARGET_COL] + TEMPORAL_FEATURE_COLS,
        "lookback":     24,
        "description":  "Load + cyclical temporal features",
    },
    {
        "name":         "E3_lags",
        "feature_cols": [TARGET_COL] + LAG_FEATURE_COLS_DEFAULT,
        "lookback":     24,
        "description":  "Load + lag_24 + lag_168",
    },
    {
        "name":         "E4_full_24",
        "feature_cols": [TARGET_COL] + WEATHER + TEMPORAL_FEATURE_COLS + LAG_FEATURE_COLS_DEFAULT,
        "lookback":     24,
        "description":  "All features, LB=24h",
    },
    {
        "name":         "E5_full_168",
        "feature_cols": [TARGET_COL] + WEATHER + TEMPORAL_FEATURE_COLS + LAG_FEATURE_COLS_DEFAULT,
        "lookback":     168,
        "description":  "All features, LB=168h",
    },
]

# Models to evaluate
ALL_MODELS = [
    "LinearRegression",
    "LSTM",
    "BiLSTM",
    "Transformer",
    "TFT",
    "TCN",
    "NBEATS-MV",
    "Informer",
    "PatchTST",
]

print("Experiment configurations:")
for e in EXPERIMENTS:
    print(f"  {e['name']:15s} | LB={e['lookback']:3d}h | {len(e['feature_cols']):2d} features | {e['description']}")
print(f"\nModels: {ALL_MODELS}")
print(f"Total combinations: {len(EXPERIMENTS)} × {len(ALL_MODELS)} = {len(EXPERIMENTS)*len(ALL_MODELS)}")

In [ ]:
# ── Cell 7: Smoke Tests ────────────────────────────────────────────────
# Verify all models build and produce correct output shapes before
# running the expensive full experiment matrix.

print("=" * 60)
print("SMOKE TESTS: Shape verification for all new architectures")
print("=" * 60)

smoke_batch = np.random.randn(4, 24, 1).astype(np.float32)  # E0 shape
smoke_batch_168 = np.random.randn(4, 168, 13).astype(np.float32)  # E5 shape
smoke_cols_full = ([TARGET_COL] + WEATHER + TEMPORAL_FEATURE_COLS + LAG_FEATURE_COLS_DEFAULT)

new_models_smoke = [
    ("Transformer",  lambda lb, nf, fc: build_transformer(lb, nf, HORIZON, **NEW_MODEL_HPARAMS["Transformer"])),
    ("TFT",          lambda lb, nf, fc: build_tft(lb, nf, HORIZON, feature_cols=fc, **NEW_MODEL_HPARAMS["TFT"])),
    ("TCN",          lambda lb, nf, fc: build_tcn(lb, nf, HORIZON, **NEW_MODEL_HPARAMS["TCN"])),
    ("NBEATS-MV",    lambda lb, nf, fc: build_nbeats(lb, nf, HORIZON, **NEW_MODEL_HPARAMS["NBEATS-MV"])),
    ("Informer",     lambda lb, nf, fc: build_informer(lb, nf, HORIZON, **NEW_MODEL_HPARAMS["Informer"])),
    ("PatchTST",     lambda lb, nf, fc: build_patchtst(lb, nf, HORIZON, **NEW_MODEL_HPARAMS["PatchTST"])),
]

smoke_results = []
for mname, builder in new_models_smoke:
    for (lb, nf, batch, fc) in [
        (24,  1,  smoke_batch,     [TARGET_COL]),
        (24,  13, np.random.randn(4, 24, 13).astype(np.float32), smoke_cols_full),
        (168, 13, smoke_batch_168, smoke_cols_full),
    ]:
        try:
            set_seed(SEED)
            m = builder(lb, nf, fc)
            out = m(batch, training=False)
            out_shape = tuple(out.shape)
            assert out_shape == (4, HORIZON), \
                f"Shape mismatch: expected (4, {HORIZON}), got {out_shape}"
            nan_check = np.any(np.isnan(out.numpy()))
            status = "✓" if not nan_check else "NaN!"
            smoke_results.append({"model": mname, "lookback": lb, "n_features": nf,
                                   "output_shape": out_shape, "has_nan": nan_check,
                                   "n_params": m.count_params(), "status": status})
            print(f"  {mname:12s} lb={lb:3d} nf={nf:2d}  out={out_shape}  "
                  f"params={m.count_params():,}  {status}")
        except Exception as e:
            print(f"  {mname:12s} lb={lb:3d} nf={nf:2d}  ERROR: {e}")
            smoke_results.append({"model": mname, "lookback": lb, "n_features": nf,
                                   "output_shape": None, "has_nan": None,
                                   "n_params": None, "status": f"ERROR: {e}"})

df_smoke = pd.DataFrame(smoke_results)
print("\nSmoke test summary:")
print(df_smoke.to_string(index=False))

failed_smoke = df_smoke[df_smoke["status"] != "✓"]
if len(failed_smoke) > 0:
    print(f"\n[WARNING] {len(failed_smoke)} smoke tests FAILED:")
    print(failed_smoke.to_string())
else:
    print("\nAll smoke tests PASSED ✓")

In [ ]:
# ── Cell 8: Run All Validation Experiments ─────────────────────────────
# Model selection is done ONLY on validation — test is held-out here.

if not RUN_EXPERIMENTS:
    print("RUN_EXPERIMENTS=False — skipping training.")
else:
    # 8a. Naive baselines
    print("=" * 60)
    print("Naive Baselines (Validation)")
    print("=" * 60)
    val_baselines = run_naive_baselines(split="val")
    for name, m in val_baselines.items():
        print(f"  {name:<20s}  MAE={m['MAE']:.2f}  MAPE={m['MAPE']:.2f}%")

    # 8b. All model × experiment combinations
    all_val_results = {}   # (exp_name, model_name) → result dict
    failed_experiments = []

    for exp in EXPERIMENTS:
        for mname in ALL_MODELS:
            print(f"\n{'='*60}")
            print(f" {exp['name']} | {mname}")
            print(f"{'='*60}")
            try:
                res = run_experiment(
                    exp["name"],
                    exp["feature_cols"],
                    exp["lookback"],
                    mname,
                    evaluate_on_test=False,  # test evaluated post-selection
                )
                all_val_results[(exp["name"], mname)] = res
            except Exception as e:
                import traceback
                print(f"  [ERROR] {exp['name']}/{mname}: {e}")
                traceback.print_exc()
                failed_experiments.append({
                    "experiment": exp["name"],
                    "model": mname,
                    "error": str(e),
                })

    print("\nValidation experiments complete.")
    if failed_experiments:
        print(f"\n[WARNING] {len(failed_experiments)} experiments failed:")
        for fe in failed_experiments:
            print(f"  {fe['experiment']}/{fe['model']}: {fe['error']}")

In [ ]:
# ── Cell 9: Model Selection & Complete Test Evaluation ─────────────────
# A. Model selection uses VALIDATION MAPE only.
# B. Complete test evaluation of ALL configurations (post-selection analysis).

print("=" * 60)
print("A. MODEL SELECTION (Validation MAPE)")
print("=" * 60)

best_key    = min(all_val_results, key=lambda k: all_val_results[k]["val_metrics"]["MAPE"])
best_result = all_val_results[best_key]
best_exp_name, best_model = best_key

print(f"  Best config : {best_exp_name} / {best_model}")
print(f"  Val MAPE    : {best_result['val_metrics']['MAPE']:.4f}%")

best_exp_cfg = next(e for e in EXPERIMENTS if e["name"] == best_exp_name)

# Final test evaluation for the BEST config
print("\nRunning FINAL TEST evaluation for best config...")
final_result = run_experiment(
    best_exp_cfg["name"],
    best_exp_cfg["feature_cols"],
    best_exp_cfg["lookback"],
    best_model,
    evaluate_on_test=True,
)

# Naive baselines on test
print("\nNaive baselines on TEST...")
test_baselines = run_naive_baselines(split="test")

print("\n" + "=" * 60)
print("FINAL TEST RESULTS (best config only)")
print("=" * 60)
tm = final_result.get("test_metrics", {})
print(f"  {best_exp_name} / {best_model}")
if tm:
    print(f"  MAE={tm['MAE']:.2f}  RMSE={tm['RMSE']:.2f}  "
          f"MAPE={tm['MAPE']:.2f}%  sMAPE={tm['sMAPE']:.2f}%")
for name, m in test_baselines.items():
    print(f"  {name:<20s}  MAE={m['MAE']:.2f}  MAPE={m['MAPE']:.2f}%")

print("\n" + "=" * 60)
print("B. COMPLETE POST-SELECTION TEST EVALUATION")
print("(For research analysis ONLY — NOT used for model selection)")
print("=" * 60)

# Load test predictions for all configurations from saved CSVs
all_test_results = {}
for exp in EXPERIMENTS:
    for mname in ALL_MODELS:
        key = (exp["name"], mname)
        if key not in all_val_results:
            continue
        # Load from saved CSVs (no retraining)
        try:
            df_te = load_predictions(exp["name"], mname, "test", exp["lookback"])
            pred_te  = df_te["pred"].values.reshape(-1, HORIZON)
            actual_te = df_te["true"].values.reshape(-1, HORIZON)
            test_mets = compute_all_metrics(actual_te, pred_te)
            all_test_results[key] = test_mets
        except Exception as e:
            print(f"  [WARNING] Cannot load test preds for {exp['name']}/{mname}: {e}")

In [ ]:
# ── Cell 10: Save Aggregated Metrics CSVs & Prediction Index ──────────

# Save all collected metric records
if METRIC_RECORDS:
    df_metrics_all = pd.DataFrame(METRIC_RECORDS)
    df_val_metrics = df_metrics_all[df_metrics_all["split"] == "validation"].copy()
    df_test_metrics = df_metrics_all[df_metrics_all["split"] == "test"].copy()

    df_val_metrics.to_csv(METRICS_DIR / "validation_metrics.csv", index=False)
    df_test_metrics.to_csv(METRICS_DIR / "test_metrics.csv", index=False)
    print(f"Saved validation_metrics.csv ({len(df_val_metrics)} rows)")
    print(f"Saved test_metrics.csv ({len(df_test_metrics)} rows)")

# Build prediction index
index_rows = []
for split in ["validation", "test"]:
    for exp in EXPERIMENTS:
        for mname in ALL_MODELS:
            pred_path = PRED_DIR / split / exp["name"] / f"{mname}_lb{exp['lookback']}.csv"
            if pred_path.exists():
                try:
                    df_p = pd.read_csv(pred_path)
                    n_windows = len(df_p) // HORIZON
                    # Recompute metrics from file
                    p_arr = df_p["pred"].values.reshape(-1, HORIZON)
                    t_arr = df_p["true"].values.reshape(-1, HORIZON)
                    mets  = compute_all_metrics(t_arr, p_arr)
                    index_rows.append({
                        "experiment":      exp["name"],
                        "model":           mname,
                        "split":           split,
                        "lookback":        exp["lookback"],
                        "prediction_file": str(pred_path.relative_to(PROJECT_ROOT)),
                        "n_forecast_windows": n_windows,
                        "overall_mae":     mets["MAE"],
                        "overall_rmse":    mets["RMSE"],
                        "overall_mape":    mets["MAPE"],
                        "overall_smape":   mets["sMAPE"],
                    })
                except Exception as e:
                    print(f"  [WARNING] Cannot index {pred_path}: {e}")

df_index = pd.DataFrame(index_rows)
df_index.to_csv(RESULTS_DIR / "prediction_index.csv", index=False)
print(f"\nSaved prediction_index.csv ({len(df_index)} entries)")
print(df_index[["experiment", "model", "split", "lookback",
                "n_forecast_windows", "overall_mape"]].to_string(index=False))

In [ ]:
# ── Cell 11: Summary Tables ────────────────────────────────────────────

exp_names   = [e["name"]  for e in EXPERIMENTS]
model_names = ALL_MODELS

# ─────────────────────────────────────────────────────────────────────
# TABLE 1 — Model/Experiment Configuration
# ─────────────────────────────────────────────────────────────────────
table1_rows = []
for exp in EXPERIMENTS:
    for mname in ALL_MODELS:
        table1_rows.append({
            "Experiment":       exp["name"],
            "Model":            mname,
            "Input Variables":  ", ".join(exp["feature_cols"][:3]) + (" ..." if len(exp["feature_cols"]) > 3 else ""),
            "Input Length (h)": exp["lookback"],
            "Output Variable":  TARGET_COL,
            "Output Length (h)": HORIZON,
            "N Features":       len(exp["feature_cols"]),
        })
df_table1 = pd.DataFrame(table1_rows)
df_table1.to_csv(RESULTS_DIR / "table1_configuration.csv", index=False)
print("TABLE 1 — Model/Experiment Configuration")
print(df_table1.drop_duplicates(subset=["Experiment"]).to_string(index=False))

# ─────────────────────────────────────────────────────────────────────
# TABLE 2 — Validation MAPE Matrix
# ─────────────────────────────────────────────────────────────────────
val_mape_matrix = pd.DataFrame(index=exp_names, columns=model_names, dtype=float)
for (en, mn), res in all_val_results.items():
    val_mape_matrix.loc[en, mn] = res["val_metrics"]["MAPE"]

# Add naive baselines
for name, m in val_baselines.items():
    val_mape_matrix.loc[name, "LinearRegression"] = m["MAPE"]

val_mape_matrix.to_csv(RESULTS_DIR / "table2_val_mape_matrix.csv")
print("\nTABLE 2 — Validation MAPE Matrix (%)")
print("=" * 80)
print(val_mape_matrix.round(4).to_string())

# ─────────────────────────────────────────────────────────────────────
# TABLE 3 — Full Validation Metrics
# ─────────────────────────────────────────────────────────────────────
val_summary_rows = []
for (en, mn), res in all_val_results.items():
    row = {"Experiment": en, "Model": mn, "Lookback": res["lookback"]}
    row.update(res["val_metrics"])
    val_summary_rows.append(row)
df_table3 = pd.DataFrame(val_summary_rows).sort_values("MAPE")
df_table3.to_csv(RESULTS_DIR / "table3_full_val_metrics.csv", index=False)
print("\nTABLE 3 — Full Validation Metrics (sorted by MAPE)")
print("=" * 80)
print(df_table3.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────
# TABLE 4 — Official Final Test Results (best config only)
# ─────────────────────────────────────────────────────────────────────
table4_rows = []
best_tm = final_result.get("test_metrics", {})
row4 = {"Experiment": best_exp_name, "Model": best_model,
         "Lookback": best_result["lookback"], "split": "test (official)"}
row4.update(best_tm)
table4_rows.append(row4)
for name, m in test_baselines.items():
    r = {"Experiment": name, "Model": "naive", "Lookback": "N/A", "split": "test (naive)"}
    r.update(m)
    table4_rows.append(r)

df_table4 = pd.DataFrame(table4_rows)
df_table4.to_csv(RESULTS_DIR / "table4_official_test_results.csv", index=False)
print("\nTABLE 4 — Official Final Test Results (best config only)")
print("=" * 80)
print(df_table4.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────
# TABLE 5 — Complete Post-selection Test Evaluation
# ─────────────────────────────────────────────────────────────────────
test_summary_rows = []
for (en, mn), test_mets in all_test_results.items():
    val_mape = all_val_results[(en, mn)]["val_metrics"]["MAPE"]
    row = {"Experiment": en, "Model": mn,
           "Lookback": all_val_results[(en, mn)]["lookback"],
           "Val_MAPE": val_mape}
    row.update({"Test_" + k: v for k, v in test_mets.items()})
    test_summary_rows.append(row)

df_table5 = pd.DataFrame(test_summary_rows).sort_values("Test_MAPE")
df_table5.to_csv(RESULTS_DIR / "table5_post_selection_test.csv", index=False)
print("\nTABLE 5 — Post-selection Test Evaluation (ALL configs)")
print("NOTE: This table is for RESEARCH ANALYSIS ONLY — NOT model selection evidence.")
print("=" * 80)
print(df_table5.to_string(index=False))

# ─────────────────────────────────────────────────────────────────────
# TABLE 6 — Architecture Comparison
# ─────────────────────────────────────────────────────────────────────
arch_rows = []
for mname in ALL_MODELS:
    model_keys = [(en, mn) for (en, mn) in all_val_results if mn == mname]
    if not model_keys:
        continue
    best_k = min(model_keys, key=lambda k: all_val_results[k]["val_metrics"]["MAPE"])
    best_en = best_k[0]
    val_m   = all_val_results[best_k]["val_metrics"]
    test_m  = all_test_results.get(best_k, {})
    n_params = all_val_results[best_k].get("n_params", None)
    arch_rows.append({
        "Architecture":       mname,
        "Best_Val_MAPE":       val_m["MAPE"],
        "Best_Experiment":    best_en,
        "Lookback":           all_val_results[best_k]["lookback"],
        "Test_MAPE":          test_m.get("MAPE", float("nan")),
        "Test_MAE":           test_m.get("MAE",  float("nan")),
        "Test_RMSE":          test_m.get("RMSE", float("nan")),
        "Test_sMAPE":         test_m.get("sMAPE",float("nan")),
        "N_Params":           n_params,
    })

df_table6 = pd.DataFrame(arch_rows).sort_values("Best_Val_MAPE")
df_table6.to_csv(RESULTS_DIR / "table6_architecture_comparison.csv", index=False)
print("\nTABLE 6 — Architecture Comparison")
print("=" * 80)
print(df_table6.to_string(index=False))

In [ ]:
# ── Cell 12: Analysis ──────────────────────────────────────────────────

print("=" * 70)
print("ANALYSIS RESULTS")
print("=" * 70)

# ── A. Feature Effect Analysis ─────────────────────────────────────────
print("\nA. FEATURE EFFECT (vs E0 baseline)")
print("-" * 50)
pairs = [("E0_load_only", "E1_weather"),
          ("E0_load_only", "E2_temporal"),
          ("E0_load_only", "E3_lags"),
          ("E0_load_only", "E4_full_24")]

for e0_name, ex_name in pairs:
    print(f"  E0 → {ex_name}:")
    for mname in ALL_MODELS:
        k0 = (e0_name, mname)
        kx = (ex_name, mname)
        if k0 in all_val_results and kx in all_val_results:
            m0 = all_val_results[k0]["val_metrics"]["MAPE"]
            mx = all_val_results[kx]["val_metrics"]["MAPE"]
            delta = mx - m0
            sign  = "▲" if delta > 0 else "▼"
            print(f"    {mname:<16s}  E0={m0:.2f}%  {ex_name.split('_')[0]+ex_name.split('_')[1]}={mx:.2f}%  "
                  f"Δ={delta:+.2f}% {sign}")

# ── B. Lookback Effect Analysis ────────────────────────────────────────
print("\nB. LOOKBACK EFFECT (E4 24h → E5 168h, same features)")
print("-" * 50)
for mname in ALL_MODELS:
    k4 = ("E4_full_24",  mname)
    k5 = ("E5_full_168", mname)
    if k4 in all_val_results and k5 in all_val_results:
        m4 = all_val_results[k4]["val_metrics"]["MAPE"]
        m5 = all_val_results[k5]["val_metrics"]["MAPE"]
        delta = m5 - m4
        sign  = "▲" if delta > 0 else "▼"
        benefit = "168h WORSE" if delta > 0 else "168h BETTER"
        print(f"  {mname:<16s}  E4={m4:.2f}%  E5={m5:.2f}%  Δ={delta:+.2f}% ({benefit})")

# ── C. Architecture Effect ─────────────────────────────────────────────
print("\nC. ARCHITECTURE EFFECT (best val MAPE per architecture, E4 full features)")
print("-" * 50)
for mname in ALL_MODELS:
    k4 = ("E4_full_24", mname)
    k5 = ("E5_full_168", mname)
    best_k = None
    if k4 in all_val_results and k5 in all_val_results:
        if all_val_results[k4]["val_metrics"]["MAPE"] < all_val_results[k5]["val_metrics"]["MAPE"]:
            best_k = k4
        else:
            best_k = k5
    elif k4 in all_val_results:
        best_k = k4
    elif k5 in all_val_results:
        best_k = k5
    if best_k:
        m = all_val_results[best_k]["val_metrics"]["MAPE"]
        print(f"  {mname:<16s}  Val MAPE={m:.2f}%  ({best_k[0]})")

# ── D. vs Naive Baselines ─────────────────────────────────────────────
print("\nD. PERFORMANCE RELATIVE TO NAIVE BASELINES (Validation)")
print("-" * 50)
weekly_naive_mape = val_baselines["weekly_naive"]["MAPE"]
print(f"  Weekly Naive MAPE: {weekly_naive_mape:.2f}%")
for mname in ALL_MODELS:
    # Use best val MAPE across all experiments for this model
    model_keys = [(en, mn) for (en, mn) in all_val_results if mn == mname]
    if not model_keys:
        continue
    best_k = min(model_keys, key=lambda k: all_val_results[k]["val_metrics"]["MAPE"])
    m = all_val_results[best_k]["val_metrics"]["MAPE"]
    pct_improv = (weekly_naive_mape - m) / weekly_naive_mape * 100
    print(f"  {mname:<16s}  Best Val MAPE={m:.2f}%  "
          f"Improvement over Weekly Naive: {pct_improv:+.1f}%")

# ── E. Key Research Questions ─────────────────────────────────────────
print("\nE. KEY RESEARCH QUESTIONS")
print("-" * 50)
# Q1: Does complexity outperform LR with rich features?
lr_e4 = all_val_results.get(("E4_full_24", "LinearRegression"), {})
if lr_e4:
    lr_mape = lr_e4["val_metrics"]["MAPE"]
    print(f"  LinearRegression E4 val MAPE: {lr_mape:.2f}%")
    for mname in [m for m in ALL_MODELS if m not in ("LinearRegression",)]:
        k4 = ("E4_full_24", mname)
        k5 = ("E5_full_168", mname)
        best_mape = None
        for k in [k4, k5]:
            if k in all_val_results:
                m_v = all_val_results[k]["val_metrics"]["MAPE"]
                if best_mape is None or m_v < best_mape:
                    best_mape = m_v
        if best_mape is not None:
            beats = "YES ✓" if best_mape < lr_mape else "NO  ✗"
            print(f"    {mname:<16s}  best={best_mape:.2f}%  "
                  f"Outperforms LR(E4): {beats}  Δ={best_mape-lr_mape:+.2f}%")

print("\nAnalysis complete.")

In [ ]:
# ── Cell 13: Visualizations ────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
COLORS = sns.color_palette("tab10", len(ALL_MODELS))
VIZ_DIR = RESULTS_DIR / "figures"
VIZ_DIR.mkdir(exist_ok=True)

# ── 13A: Validation MAPE Heatmap (full matrix) ────────────────────────
numeric_matrix = val_mape_matrix.loc[
    [e["name"] for e in EXPERIMENTS],
    ALL_MODELS
].dropna(how="all").astype(float)

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    numeric_matrix,
    annot=True, fmt=".2f", cmap="RdYlGn_r",
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "MAPE (%)"},
)
ax.set_title("Validation MAPE Heatmap — All Experiments × All Architectures", fontsize=14)
ax.set_xlabel("Architecture")
ax.set_ylabel("Experiment")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(VIZ_DIR / "val_mape_heatmap_full.png", dpi=150)
plt.show()

# ── 13B: Architecture Comparison Bar Chart (E4 & E5) ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, exp_name, title in zip(axes,
    ["E4_full_24", "E5_full_168"],
    ["E4 (LB=24h, Full Features)", "E5 (LB=168h, Full Features)"]):

    mapes = []
    models_plot = []
    for mname in ALL_MODELS:
        k = (exp_name, mname)
        if k in all_val_results:
            mapes.append(all_val_results[k]["val_metrics"]["MAPE"])
            models_plot.append(mname)

    bars = ax.bar(models_plot, mapes, color=COLORS[:len(models_plot)], edgecolor="black")
    ax.axhline(val_baselines["weekly_naive"]["MAPE"], color="red",
               linestyle="--", linewidth=1.5, label="Weekly Naive")
    ax.set_title(title, fontsize=13)
    ax.set_ylabel("Validation MAPE (%)")
    ax.tick_params(axis="x", rotation=30)
    ax.legend()
    # Annotate bars
    for bar, v in zip(bars, mapes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Architecture Comparison — Validation MAPE", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(VIZ_DIR / "architecture_comparison_E4_E5.png", dpi=150)
plt.show()

# ── 13C: Lookback Effect (E4 vs E5) per architecture ─────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(ALL_MODELS))
w = 0.35
e4_mapes = []
e5_mapes = []
for mname in ALL_MODELS:
    k4 = ("E4_full_24",  mname)
    k5 = ("E5_full_168", mname)
    e4_mapes.append(all_val_results[k4]["val_metrics"]["MAPE"] if k4 in all_val_results else np.nan)
    e5_mapes.append(all_val_results[k5]["val_metrics"]["MAPE"] if k5 in all_val_results else np.nan)

ax.bar(x - w/2, e4_mapes, w, label="E4 (LB=24h)",  color="#4c72b0", edgecolor="black")
ax.bar(x + w/2, e5_mapes, w, label="E5 (LB=168h)", color="#dd8452", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(ALL_MODELS, rotation=30, ha="right")
ax.set_ylabel("Validation MAPE (%)")
ax.set_title("Lookback Effect: E4 (24h) vs E5 (168h) — Same Features", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(VIZ_DIR / "lookback_effect_E4_vs_E5.png", dpi=150)
plt.show()

# ── 13D: Feature effect on best model ────────────────────────────────
best_m = best_model
feat_mapes = []
feat_labels = []
for exp in EXPERIMENTS[:5]:  # E0–E4 (all 24h)
    k = (exp["name"], best_m)
    if k in all_val_results:
        feat_mapes.append(all_val_results[k]["val_metrics"]["MAPE"])
        feat_labels.append(exp["name"])

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(feat_labels, feat_mapes, color=sns.color_palette("Set2", len(feat_labels)))
ax.set_title(f"Feature Effect — {best_m} (Val MAPE, LB=24h)", fontsize=13)
ax.set_ylabel("Validation MAPE (%)")
plt.tight_layout()
plt.savefig(VIZ_DIR / f"feature_effect_{best_m}.png", dpi=150)
plt.show()

# ── 13E: Best model test forecast (first 7 days) ─────────────────────
te_preds_key = (best_exp_name, best_model)
if te_preds_key in all_val_results and "pred_te_mw" in all_val_results[te_preds_key]:
    actual_te = all_val_results[te_preds_key]["actual_te_mw"]
    pred_te   = all_val_results[te_preds_key]["pred_te_mw"]
elif "actual_te_mw" in final_result:
    actual_te = final_result["actual_te_mw"]
    pred_te   = final_result["pred_te_mw"]
else:
    df_te_csv = load_predictions(best_exp_name, best_model, "test",
                                  best_exp_cfg["lookback"])
    actual_te = df_te_csv["true"].values.reshape(-1, HORIZON)
    pred_te   = df_te_csv["pred"].values.reshape(-1, HORIZON)

n_plot = min(168, len(actual_te))
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(actual_te[:n_plot, 0], label="True Load", color="black", linewidth=2)
ax.plot(pred_te[:n_plot, 0],
        label=f"Predicted ({best_exp_name}/{best_model})",
        color="#1f77b4", linestyle="--", linewidth=1.5)
ax.set_title(f"Test Set — Best Config (h=1, First 7 Days)  [{best_exp_name}/{best_model}]",
             fontsize=13)
ax.set_xlabel("Window Index")
ax.set_ylabel("Load (MW)")
ax.legend()
plt.tight_layout()
plt.savefig(VIZ_DIR / "best_config_test_forecast.png", dpi=150)
plt.show()

# ── 13F: Per-window MAPE distribution (best model) ────────────────────
df_best_pred = load_predictions(best_exp_name, best_model, "test",
                                  best_exp_cfg["lookback"])
window_mapes_series = df_best_pred.groupby("forecast_origin")["window_mape"].first()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(window_mapes_series.values, color="#ff7f0e", linewidth=0.8, alpha=0.85)
p90 = window_mapes_series.quantile(0.90)
axes[0].axhline(p90, color="red", linestyle="--", linewidth=1.5,
                label=f"90th Percentile ({p90:.2f}%)")
axes[0].set_title(f"Per-Window MAPE Over Time — {best_model}", fontsize=12)
axes[0].set_xlabel("Window Index")
axes[0].set_ylabel("Window MAPE (%)")
axes[0].legend()

axes[1].hist(window_mapes_series.values, bins=50, color="#2ca02c", edgecolor="black", alpha=0.8)
axes[1].set_title(f"Window MAPE Distribution — {best_model}", fontsize=12)
axes[1].set_xlabel("Window MAPE (%)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(VIZ_DIR / "window_mape_distribution.png", dpi=150)
plt.show()

print(f"All figures saved to: {VIZ_DIR}")

In [ ]:
# ── Cell 14: Hyperparameter Documentation ─────────────────────────────
print("=" * 70)
print("HYPERPARAMETER TABLE")
print("=" * 70)

hparam_rows = []

# Existing models
hparam_rows.append({"Model": "LinearRegression", "Parameter": "No trainable hyperparameters", "Value": "N/A"})
for mname, hd in [
    ("LSTM",   {"units": UNITS, "dropout": DROPOUT, "learning_rate": LR, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "patience": PATIENCE}),
    ("BiLSTM", {"units": UNITS, "dropout": DROPOUT, "learning_rate": LR, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "patience": PATIENCE}),
]:
    for k, v in hd.items():
        hparam_rows.append({"Model": mname, "Parameter": k, "Value": v})

# New models
for mname, hd in NEW_MODEL_HPARAMS.items():
    for k, v in hd.items():
        hparam_rows.append({"Model": mname, "Parameter": k, "Value": v})
    # Common training hparams
    for k, v in [("epochs", EPOCHS), ("batch_size", BATCH_SIZE), ("patience", PATIENCE)]:
        hparam_rows.append({"Model": mname, "Parameter": k, "Value": v})

df_hparams = pd.DataFrame(hparam_rows)
df_hparams.to_csv(RESULTS_DIR / "hyperparameters.csv", index=False)
print(df_hparams.to_string(index=False))

# ── Complexity Summary ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("COMPUTATIONAL COMPLEXITY (E4 configuration, 13 features, LB=24h)")
print("=" * 70)
complexity_rows = []
for mname in ALL_MODELS:
    k4 = ("E4_full_24", mname)
    if k4 in all_val_results:
        res = all_val_results[k4]
        complexity_rows.append({
            "Model":         mname,
            "N_Params":      res.get("n_params", "N/A"),
            "Train_Time_s":  f"{res.get('train_time_s', 0):.1f}" if res.get('train_time_s') else "N/A",
            "Best_Val_MAPE": f"{res['val_metrics']['MAPE']:.4f}%",
        })

df_complexity = pd.DataFrame(complexity_rows)
df_complexity.to_csv(RESULTS_DIR / "complexity_summary.csv", index=False)
print(df_complexity.to_string(index=False))

In [ ]:
# ── Cell 15: Final Summary ─────────────────────────────────────────────
print("=" * 70)
print("EXPERIMENT 04 — COMPLETE")
print("=" * 70)

# List all saved files
print("\nSaved files:")
for f in sorted(RESULTS_DIR.rglob("*.csv")):
    size_kb = f.stat().st_size / 1024
    print(f"  {str(f.relative_to(PROJECT_ROOT)):70s}  {size_kb:.1f} KB")

print("\nFigures:")
for f in sorted(VIZ_DIR.glob("*.png")):
    print(f"  {f.name}")

print(f"\n{'='*70}")
print(f"Best configuration (by Validation MAPE):")
print(f"  Experiment : {best_exp_name}")
print(f"  Model      : {best_model}")
vm = best_result["val_metrics"]
print(f"  Val MAE={vm['MAE']:.2f}  RMSE={vm['RMSE']:.2f}  MAPE={vm['MAPE']:.2f}%  sMAPE={vm['sMAPE']:.2f}%")
if "test_metrics" in final_result:
    tm = final_result["test_metrics"]
    print(f"  Test MAE={tm['MAE']:.2f}  RMSE={tm['RMSE']:.2f}  MAPE={tm['MAPE']:.2f}%  sMAPE={tm['sMAPE']:.2f}%")

if 'failed_experiments' in dir() and failed_experiments:
    print(f"\n⚠️  Failed experiments ({len(failed_experiments)}):")
    for fe in failed_experiments:
        print(f"  {fe['experiment']}/{fe['model']}: {fe['error']}")

print("\nDone.")